
# Urban Mobility Intelligence

## 01 — Data Ingestion and Inspection

### Project Objective
To develop a scalable urban mobility analytics pipeline using PySpark and Databricks, investigating journey patterns, operational behaviour and unusual observations within a synthetic metropolitan transportation dataset.

### Notebook Objective
Establish the analytical environment, load the source dataset, and inspect its structure, schema and initial characteristics before performing data quality assessments.


In [0]:
# Inspect the Spark environment

print("Spark Version:", spark.version)
print("Spark Session:", spark)

Spark Version: 4.2.0
Spark Session: <pyspark.sql.connect.session.SparkSession object at 0xffb50e08dc40>


In [0]:
# List files stored in our Unity Catalog volume

display(dbutils.fs.ls("/Volumes/workspace/default/urban_mobility/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/urban_mobility/data_dictionary.csv,data_dictionary.csv,3274,1795305058000
dbfs:/Volumes/workspace/default/urban_mobility/urban_mobility_trips.csv,urban_mobility_trips.csv,29225784,1795305066000
dbfs:/Volumes/workspace/default/urban_mobility/zone_reference.csv,zone_reference.csv,653,1795305058000


In [0]:
# Define the source file path

trips_path = "/Volumes/workspace/default/urban_mobility/urban_mobility_trips.csv"

# Read the CSV file into a Spark DataFrame

trips_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(trips_path)
)

print("Dataset successfully loaded.")

Dataset successfully loaded.


In [0]:
# Inspect the dataset structure

print("Number of rows:", trips_df.count())
print("Number of columns:", len(trips_df.columns))

# Display column names and data types
trips_df.printSchema()

Number of rows: 120000
Number of columns: 26
root
 |-- trip_id: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- fare_amount_eur: double (nullable = true)
 |-- toll_amount_eur: double (nullable = true)
 |-- tip_amount_eur: double (nullable = true)
 |-- total_amount_eur: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- service_type: string (nullable = true)
 |-- dispatch_channel: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- scenario_label: string (nullable = true)
 |-- scenario_detail: string (nullable = true)
 |-- pickup_date: date (nul

In [0]:
# Preview the first 10 records

display(trips_df.limit(10))

trip_id,pickup_datetime,dropoff_datetime,driver_id,vehicle_id,pickup_zone,dropoff_zone,trip_distance_km,trip_duration_min,fare_amount_eur,toll_amount_eur,tip_amount_eur,total_amount_eur,payment_method,passenger_count,service_type,dispatch_channel,weather_condition,scenario_label,scenario_detail,pickup_date,pickup_hour,pickup_day_of_week,average_speed_kmh,fare_per_km_eur,tip_to_fare_ratio
TRP0000001,2025-09-01T00:00:04.000Z,2025-09-01T00:08:05.876Z,DRV0044,VEH0130,Outer North,South Residential,2.55,8.0,10.12,0.0,0.0,10.12,Card,1,Standard,Mobile App,Rain,Routine,No injected anomaly,2025-09-01,0,Monday,19.12,3.97,0.0
TRP0000002,2025-09-01T00:00:25.000Z,2025-09-01T00:22:21.484Z,DRV0090,VEH0114,West Shopping District,Convention Centre,8.61,21.9,23.62,0.0,0.0,23.62,Card,1,Standard,Phone Dispatch,Clear,Routine,No injected anomaly,2025-09-01,0,Monday,23.59,2.74,0.0
TRP0000003,2025-09-01T00:02:23.000Z,2025-09-01T00:18:53.293Z,DRV0190,VEH0149,East Medical Campus,Harbourfront,5.81,16.5,18.41,0.0,0.0,18.41,Card,2,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-09-01,0,Monday,21.13,3.17,0.0
TRP0000004,2025-09-01T00:03:04.000Z,2025-09-01T00:11:25.727Z,DRV0008,VEH0297,University District,Financial Quarter,4.12,8.4,13.24,0.0,2.25,15.5,Mobile Wallet,1,Standard,Mobile App,Cloudy,Routine,No injected anomaly,2025-09-01,0,Monday,29.43,3.21,0.17
TRP0000005,2025-09-01T00:03:51.000Z,2025-09-01T00:06:51.000Z,DRV0127,VEH0081,Central Business District,Central Station,2.7,3.0,21.37,0.0,0.16,21.53,Card,1,Standard,Mobile App,Rain,Review,Fare-to-distance outlier,2025-09-01,0,Monday,54.0,7.91,0.007
TRP0000006,2025-09-01T00:04:41.000Z,2025-09-01T00:17:11.115Z,DRV0119,VEH0037,Harbourfront,Financial Quarter,5.39,12.5,14.59,0.0,0.47,15.06,Card,1,Standard,Mobile App,Cloudy,Routine,No injected anomaly,2025-09-01,0,Monday,25.87,2.71,0.032
TRP0000007,2025-09-01T00:04:52.000Z,2025-09-01T00:14:14.807Z,DRV0027,VEH0010,Central Business District,University District,2.48,9.4,9.97,0.0,0.51,10.48,Card,1,Standard,Corporate Booking,Clear,Routine,No injected anomaly,2025-09-01,0,Monday,15.83,4.02,0.051
TRP0000008,2025-09-01T00:04:58.000Z,2025-09-01T00:12:17.971Z,DRV0017,VEH0008,Outer North,East Medical Campus,3.39,7.3,10.89,0.0,2.92,13.81,Card,1,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-09-01,0,Monday,27.86,3.21,0.268
TRP0000009,2025-09-01T00:05:08.000Z,2025-09-01T00:15:07.805Z,DRV0251,VEH0345,East Medical Campus,Financial Quarter,2.03,10.0,9.19,0.0,0.74,9.93,Mobile Wallet,1,Standard,Mobile App,Cloudy,Routine,No injected anomaly,2025-09-01,0,Monday,12.18,4.53,0.081
TRP0000010,2025-09-01T00:05:20.000Z,2025-09-01T00:12:21.698Z,DRV0223,VEH0067,Old Town,Central Business District,3.19,7.0,11.0,0.0,2.4,13.4,Card,2,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-09-01,0,Monday,27.34,3.45,0.218


In [0]:
# Generate descriptive statistics for numerical columns

trips_df.describe(
    "trip_distance_km",
    "trip_duration_min",
    "fare_amount_eur",
    "total_amount_eur",
    "passenger_count",
    "average_speed_kmh"
).show()

+-------+-----------------+------------------+------------------+------------------+------------------+------------------+
|summary| trip_distance_km| trip_duration_min|   fare_amount_eur|  total_amount_eur|   passenger_count| average_speed_kmh|
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+
|  count|           120000|            120000|            120000|            120000|            120000|            120000|
|   mean|5.938227833333345|18.338734999999993|18.994730499999996|20.961922666666716|1.6409916666666666|21.948083499999978|
| stddev|5.142825416364286|15.703609790780979|13.135145267920711|14.986369464247632|0.9931748267882444|13.598738637748681|
|    min|             0.35|               2.0|               5.0|               5.0|                 1|              0.98|
|    max|            120.0|             422.5|            251.95|            459.52|                 6|            717.09|
+-------+-------

### Initial Data Observations

- The dataset contains 120,000 trip-level records
  across 26 attributes.
- The inferred schema is consistent with the expected
  data types.
- All six inspected numerical variables contain
  120,000 non-null observations.
- Average trip distance is 5.94 km, with an average
  duration of 18.34 minutes.
- Extreme observations exist across trip distance,
  duration, financial amounts and average speed.
- These observations require further investigation
  before determining whether they represent data
  quality issues or operational anomalies.
- No records have been removed at this stage.

## Data Quality and Validation

This section evaluates the completeness, uniqueness,
consistency and validity of the source dataset before
exploratory analysis and anomaly detection.

In [0]:
from pyspark.sql import functions as F

# Count missing values across all columns

missing_values = trips_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in trips_df.columns
])

display(missing_values)

trip_id,pickup_datetime,dropoff_datetime,driver_id,vehicle_id,pickup_zone,dropoff_zone,trip_distance_km,trip_duration_min,fare_amount_eur,toll_amount_eur,tip_amount_eur,total_amount_eur,payment_method,passenger_count,service_type,dispatch_channel,weather_condition,scenario_label,scenario_detail,pickup_date,pickup_hour,pickup_day_of_week,average_speed_kmh,fare_per_km_eur,tip_to_fare_ratio
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Duplicate and Identifier Validation

Validate the uniqueness of records and trip identifiers
to establish the analytical grain of the dataset and
prevent inaccurate aggregations.

In [0]:
# Check for complete duplicate records

total_rows = trips_df.count()
distinct_rows = trips_df.distinct().count()

print("Total records:", total_rows)
print("Distinct records:", distinct_rows)
print("Duplicate records:", total_rows - distinct_rows)

Total records: 120000
Distinct records: 120000
Duplicate records: 0


In [0]:
# Validate the uniqueness of trip identifiers

unique_trip_ids = trips_df.select("trip_id").distinct().count()

print("Total records:", total_rows)
print("Unique trip IDs:", unique_trip_ids)
print("Excess duplicate trip IDs:", total_rows - unique_trip_ids)

Total records: 120000
Unique trip IDs: 120000
Excess duplicate trip IDs: 0


In [0]:
# Identify the operational coverage of the dataset

trips_df.agg(
    F.countDistinct("trip_id").alias("unique_trips"),
    F.countDistinct("driver_id").alias("unique_drivers"),
    F.countDistinct("vehicle_id").alias("unique_vehicles"),
    F.countDistinct("pickup_zone").alias("pickup_zones"),
    F.countDistinct("dropoff_zone").alias("dropoff_zones")
).show()

+------------+--------------+---------------+------------+-------------+
|unique_trips|unique_drivers|unique_vehicles|pickup_zones|dropoff_zones|
+------------+--------------+---------------+------------+-------------+
|      120000|           300|            360|          20|           20|
+------------+--------------+---------------+------------+-------------+



### Temporal Consistency Validation

Assess the temporal coverage of the dataset and
validate the chronological consistency of trip events
and derived temporal attributes.

In [0]:
# Establish the dataset observation period

trips_df.agg(
    F.min("pickup_datetime").alias("earliest_pickup"),
    F.max("pickup_datetime").alias("latest_pickup"),
    F.max("dropoff_datetime").alias("latest_dropoff"),
    F.countDistinct("pickup_date").alias("days_observed")
).show(truncate=False)

+-------------------+-------------------+--------------------------+-------------+
|earliest_pickup    |latest_pickup      |latest_dropoff            |days_observed|
+-------------------+-------------------+--------------------------+-------------+
|2025-09-01 00:00:04|2025-11-30 23:54:51|2025-12-01 00:13:38.158166|91           |
+-------------------+-------------------+--------------------------+-------------+



In [0]:
# Identify trips with invalid chronological order

invalid_temporal_records = trips_df.filter(
    F.col("dropoff_datetime") <= F.col("pickup_datetime")
)

print("Invalid temporal records:",
      invalid_temporal_records.count())

Invalid temporal records: 0


In [0]:
# Independently calculate trip duration

duration_validation_df = trips_df.withColumn(
    "calculated_duration_min",
    (
        F.unix_timestamp("dropoff_datetime") -
        F.unix_timestamp("pickup_datetime")
    ) / 60
).withColumn(
    "duration_difference",
    F.abs(
        F.col("trip_duration_min") -
        F.col("calculated_duration_min")
    )
)

display(
    duration_validation_df.select(
        "trip_id",
        "trip_duration_min",
        "calculated_duration_min",
        "duration_difference"
    ).limit(10)
)

trip_id,trip_duration_min,calculated_duration_min,duration_difference
TRP0000001,8.0,8.016666666666667,0.016666666666667496
TRP0000002,21.9,21.933333333333334,0.03333333333333499
TRP0000003,16.5,16.5,0.0
TRP0000004,8.4,8.35,0.05000000000000071
TRP0000005,3.0,3.0,0.0
TRP0000006,12.5,12.5,0.0
TRP0000007,9.4,9.366666666666667,0.033333333333333215
TRP0000008,7.3,7.316666666666666,0.016666666666666607
TRP0000009,10.0,9.983333333333333,0.016666666666667496
TRP0000010,7.0,7.016666666666667,0.016666666666666607


In [0]:
# Summarise duration discrepancies

duration_validation_df.agg(
    F.min("duration_difference").alias("min_difference"),
    F.avg("duration_difference").alias("avg_difference"),
    F.max("duration_difference").alias("max_difference")
).show()

+--------------+-------------------+-------------------+
|min_difference|     avg_difference|     max_difference|
+--------------+-------------------+-------------------+
|           0.0|0.02341319444444371|0.05000000000001137|
+--------------+-------------------+-------------------+



In [0]:
# Identify trips with duration discrepancies of at least 1 minute

duration_discrepancies_df = duration_validation_df.filter(
    F.col("duration_difference") >= 1
)

display(
    duration_discrepancies_df.select(
        "trip_id",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_min",
        "calculated_duration_min",
        "duration_difference"
    ).orderBy(F.col("duration_difference").desc())
)

trip_id,pickup_datetime,dropoff_datetime,trip_duration_min,calculated_duration_min,duration_difference


#### Duration Validation Findings

The recorded trip durations were independently validated
against the difference between pickup and drop-off timestamps.

The maximum observed discrepancy was 0.05 minutes (3 seconds),
consistent with rounding precision.

No records exhibited duration discrepancies of one minute
or greater.

The original trip duration attribute is therefore retained
for subsequent analysis without modification.

In [0]:
# Independently derive temporal attributes

temporal_validation_df = (
    trips_df
    .withColumn(
        "calculated_pickup_date",
        F.to_date("pickup_datetime")
    )
    .withColumn(
        "calculated_pickup_hour",
        F.hour("pickup_datetime")
    )
    .withColumn(
        "calculated_pickup_day",
        F.date_format("pickup_datetime", "EEEE")
    )
)

display(
    temporal_validation_df.select(
        "pickup_datetime",
        "pickup_date",
        "calculated_pickup_date",
        "pickup_hour",
        "calculated_pickup_hour",
        "pickup_day_of_week",
        "calculated_pickup_day"
    ).limit(10)
)

pickup_datetime,pickup_date,calculated_pickup_date,pickup_hour,calculated_pickup_hour,pickup_day_of_week,calculated_pickup_day
2025-09-01T00:00:04.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:00:25.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:02:23.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:03:04.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:03:51.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:04:41.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:04:52.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:04:58.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:05:08.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday
2025-09-01T00:05:20.000Z,2025-09-01,2025-09-01,0,0,Monday,Monday


In [0]:
# Count inconsistent temporal attributes

temporal_validation_df.agg(
    F.sum(
        F.when(
            F.col("pickup_date") != F.col("calculated_pickup_date"),
            1
        ).otherwise(0)
    ).alias("date_mismatches"),

    F.sum(
        F.when(
            F.col("pickup_hour") != F.col("calculated_pickup_hour"),
            1
        ).otherwise(0)
    ).alias("hour_mismatches"),

    F.sum(
        F.when(
            F.col("pickup_day_of_week") != F.col("calculated_pickup_day"),
            1
        ).otherwise(0)
    ).alias("weekday_mismatches")
).show()

+---------------+---------------+------------------+
|date_mismatches|hour_mismatches|weekday_mismatches|
+---------------+---------------+------------------+
|              0|              0|                 0|
+---------------+---------------+------------------+



### Financial Reconciliation

Validate the integrity of transaction amounts by
independently calculating total charges from fare,
toll and tip components.

Assess financial discrepancies and identify records
that violate expected monetary constraints.

In [0]:
# Recalculate total transaction amounts

financial_validation_df = (
    trips_df
    .withColumn(
        "calculated_total_amount",
        F.round(
            F.col("fare_amount_eur") +
            F.col("toll_amount_eur") +
            F.col("tip_amount_eur"),
            2
        )
    )
    .withColumn(
        "amount_difference",
        F.round(
            F.abs(
                F.col("total_amount_eur") -
                F.col("calculated_total_amount")
            ),
            2
        )
    )
)

display(
    financial_validation_df.select(
        "trip_id",
        "fare_amount_eur",
        "toll_amount_eur",
        "tip_amount_eur",
        "total_amount_eur",
        "calculated_total_amount",
        "amount_difference"
    ).limit(10)
)

trip_id,fare_amount_eur,toll_amount_eur,tip_amount_eur,total_amount_eur,calculated_total_amount,amount_difference
TRP0000001,10.12,0.0,0.0,10.12,10.12,0.0
TRP0000002,23.62,0.0,0.0,23.62,23.62,0.0
TRP0000003,18.41,0.0,0.0,18.41,18.41,0.0
TRP0000004,13.24,0.0,2.25,15.5,15.49,0.01
TRP0000005,21.37,0.0,0.16,21.53,21.53,0.0
TRP0000006,14.59,0.0,0.47,15.06,15.06,0.0
TRP0000007,9.97,0.0,0.51,10.48,10.48,0.0
TRP0000008,10.89,0.0,2.92,13.81,13.81,0.0
TRP0000009,9.19,0.0,0.74,9.93,9.93,0.0
TRP0000010,11.0,0.0,2.4,13.4,13.4,0.0


In [0]:
# Identify transaction amount discrepancies

financial_discrepancies_df = financial_validation_df.filter(
    F.col("amount_difference") >= 0.01
)

print(
    "Financial discrepancies:",
    financial_discrepancies_df.count()
)

display(
    financial_discrepancies_df.orderBy(
        F.col("amount_difference").desc()
    )
)

Financial discrepancies: 17138


trip_id,pickup_datetime,dropoff_datetime,driver_id,vehicle_id,pickup_zone,dropoff_zone,trip_distance_km,trip_duration_min,fare_amount_eur,toll_amount_eur,tip_amount_eur,total_amount_eur,payment_method,passenger_count,service_type,dispatch_channel,weather_condition,scenario_label,scenario_detail,pickup_date,pickup_hour,pickup_day_of_week,average_speed_kmh,fare_per_km_eur,tip_to_fare_ratio,calculated_total_amount,amount_difference
TRP0066001,2025-10-21T00:50:39.000Z,2025-10-21T01:24:45.410Z,DRV0113,VEH0138,Central Station,Outer North,16.8,34.1,41.4,0.0,6.85,48.26,Card,1,Standard,Mobile App,Cloudy,Routine,No injected anomaly,2025-10-21,0,Tuesday,29.56,2.46,0.165,48.25,0.01
TRP0049233,2025-10-08T05:33:14.000Z,2025-10-08T05:43:18.453Z,DRV0047,VEH0150,Central Business District,Technology Park,2.2,10.1,9.16,0.0,0.91,10.06,Card,4,Standard,Phone Dispatch,Clear,Routine,No injected anomaly,2025-10-08,5,Wednesday,13.07,4.16,0.099,10.07,0.01
TRP0064063,2025-10-19T13:17:08.000Z,2025-10-19T14:14:56.437Z,DRV0065,VEH0137,Airport,Central Business District,36.27,57.8,87.78,2.5,12.5,102.77,Card,1,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-10-19,13,Sunday,37.65,2.42,0.142,102.78,0.01
TRP0034042,2025-09-26T16:12:00.000Z,2025-09-26T16:24:41.221Z,DRV0219,VEH0116,West Shopping District,South Residential,6.16,12.7,17.81,0.0,0.91,18.73,Card,4,Accessible,Mobile App,Clear,Routine,No injected anomaly,2025-09-26,16,Friday,29.1,2.89,0.051,18.72,0.01
TRP0060197,2025-10-16T13:54:37.000Z,2025-10-16T14:20:02.333Z,DRV0132,VEH0345,South Residential,Harbourfront,7.32,25.4,22.54,0.0,9.19,31.72,Card,1,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-10-16,13,Thursday,17.29,3.08,0.408,31.73,0.01
TRP0057301,2025-10-14T08:17:47.000Z,2025-10-14T08:55:23.098Z,DRV0255,VEH0233,University District,Financial Quarter,8.43,37.6,28.32,0.0,1.34,29.67,Mobile Wallet,3,Standard,Mobile App,Cloudy,Routine,No injected anomaly,2025-10-14,8,Tuesday,13.45,3.36,0.047,29.66,0.01
TRP0071969,2025-10-25T12:19:46.000Z,2025-10-25T12:53:52.912Z,DRV0130,VEH0254,West Shopping District,South Residential,13.1,34.1,35.96,0.0,7.39,43.36,Cash,2,Standard,Mobile App,Heavy Rain,Routine,No injected anomaly,2025-10-25,12,Saturday,23.05,2.75,0.206,43.35,0.01
TRP0069898,2025-10-23T23:28:12.000Z,2025-10-23T23:39:52.456Z,DRV0217,VEH0208,West Shopping District,North Residential,3.56,11.7,13.06,0.0,1.82,14.87,Card,1,Standard,Phone Dispatch,Clear,Routine,No injected anomaly,2025-10-23,23,Thursday,18.26,3.67,0.139,14.88,0.01
TRP0031260,2025-09-24T16:06:19.000Z,2025-09-24T16:23:57.696Z,DRV0058,VEH0002,University District,South Residential,6.77,17.6,20.91,0.0,3.25,24.17,Card,1,Standard,Street Hail,Heavy Rain,Routine,No injected anomaly,2025-09-24,16,Wednesday,23.08,3.09,0.155,24.16,0.01
TRP0031317,2025-09-24T16:56:22.000Z,2025-09-24T17:05:36.721Z,DRV0045,VEH0069,Technology Park,Central Business District,1.19,9.2,7.66,0.0,0.36,8.03,Card,2,Standard,Mobile App,Clear,Routine,No injected anomaly,2025-09-24,16,Wednesday,7.76,6.44,0.047,8.02,0.01


In [0]:
# Validate monetary constraints

trips_df.agg(
    F.sum(
        F.when(F.col("fare_amount_eur") <= 0, 1)
        .otherwise(0)
    ).alias("invalid_fares"),

    F.sum(
        F.when(F.col("toll_amount_eur") < 0, 1)
        .otherwise(0)
    ).alias("negative_tolls"),

    F.sum(
        F.when(F.col("tip_amount_eur") < 0, 1)
        .otherwise(0)
    ).alias("negative_tips"),

    F.sum(
        F.when(F.col("total_amount_eur") <= 0, 1)
        .otherwise(0)
    ).alias("invalid_totals")
).show()

+-------------+--------------+-------------+--------------+
|invalid_fares|negative_tolls|negative_tips|invalid_totals|
+-------------+--------------+-------------+--------------+
|            0|             0|            0|             0|
+-------------+--------------+-------------+--------------+



In [0]:
# Summarise financial discrepancies

financial_discrepancies_df.agg(
    F.count("*").alias("affected_records"),
    F.min("amount_difference").alias("min_difference"),
    F.round(
        F.avg("amount_difference"), 4
    ).alias("avg_difference"),
    F.max("amount_difference").alias("max_difference"),
    F.round(
        F.sum("amount_difference"), 2
    ).alias("total_absolute_difference")
).show()

+----------------+--------------+--------------+--------------+-------------------------+
|affected_records|min_difference|avg_difference|max_difference|total_absolute_difference|
+----------------+--------------+--------------+--------------+-------------------------+
|           17138|          0.01|          0.01|          0.01|                   171.38|
+----------------+--------------+--------------+--------------+-------------------------+



In [0]:
# Analyse the distribution of financial discrepancies

financial_discrepancies_df.groupBy(
    "amount_difference"
).agg(
    F.count("*").alias("affected_records")
).orderBy(
    F.col("affected_records").desc()
).show(20)

+-----------------+----------------+
|amount_difference|affected_records|
+-----------------+----------------+
|             0.01|           17138|
+-----------------+----------------+



In [0]:
# Calculate signed financial discrepancies

financial_direction_df = financial_discrepancies_df.withColumn(
    "signed_difference",
    F.round(
        F.col("total_amount_eur") -
        F.col("calculated_total_amount"),
        2
    )
)

financial_direction_df.groupBy(
    "signed_difference"
).count().orderBy(
    "signed_difference"
).show(20)

+-----------------+-----+
|signed_difference|count|
+-----------------+-----+
|            -0.01| 8521|
|             0.01| 8617|
+-----------------+-----+



In [0]:
# Define the accepted reconciliation tolerance

financial_tolerance = 0.01

# Identify discrepancies exceeding the accepted tolerance

material_discrepancies_df = financial_validation_df.filter(
    F.col("amount_difference") > financial_tolerance
)

print(
    "Discrepancies exceeding tolerance:",
    material_discrepancies_df.count()
)

Discrepancies exceeding tolerance: 0


In [0]:
# Classify financial reconciliation outcomes

financial_validation_df = financial_validation_df.withColumn(
    "reconciliation_status",
    F.when(
        F.col("amount_difference") == 0,
        "Exact Match"
    ).when(
        F.col("amount_difference") <= financial_tolerance,
        "Within Tolerance"
    ).otherwise(
        "Outside Tolerance"
    )
)

financial_validation_df.groupBy(
    "reconciliation_status"
).count().show()

+---------------------+------+
|reconciliation_status| count|
+---------------------+------+
|     Within Tolerance| 17138|
|          Exact Match|102862|
+---------------------+------+



#### Financial Reconciliation Findings

Financial reconciliation was performed across all
120,000 trip transactions.

- 102,862 transactions (85.72%) reconciled exactly.
- 17,138 transactions (14.28%) exhibited a discrepancy
  of exactly EUR 0.01.
- 8,521 transactions were understated by EUR 0.01.
- 8,617 transactions were overstated by EUR 0.01.
- The total absolute discrepancy was EUR 171.38,
  with a net signed difference of EUR 0.96.
- No violations of the defined monetary constraints
  were identified.

The magnitude and near-symmetric distribution of
the discrepancies suggest independent rounding
during synthetic data generation.

An analytical reconciliation tolerance of EUR 0.01
was adopted to preserve the original transaction
records.

No records were removed or modified.

### Derived Feature Validation

Independently recalculate numerical features to assess
their consistency with the underlying trip attributes.

The validated features will support behavioural analysis
and subsequent statistical and machine learning-based
anomaly detection.

In [0]:
# Independently calculate derived numerical features

feature_validation_df = (
    trips_df
    .withColumn(
        "calculated_speed",
        F.col("trip_distance_km") /
        (F.col("trip_duration_min") / 60)
    )
    .withColumn(
        "calculated_fare_per_km",
        F.col("fare_amount_eur") /
        F.col("trip_distance_km")
    )
    .withColumn(
        "calculated_tip_ratio",
        F.col("tip_amount_eur") /
        F.col("fare_amount_eur")
    )
)

In [0]:
# Compare original and independently calculated features

feature_validation_df = (
    feature_validation_df
    .withColumn(
        "speed_difference",
        F.abs(
            F.col("average_speed_kmh") -
            F.col("calculated_speed")
        )
    )
    .withColumn(
        "fare_per_km_difference",
        F.abs(
            F.col("fare_per_km_eur") -
            F.col("calculated_fare_per_km")
        )
    )
    .withColumn(
        "tip_ratio_difference",
        F.abs(
            F.col("tip_to_fare_ratio") -
            F.col("calculated_tip_ratio")
        )
    )
)


In [0]:

# Summarise feature discrepancies across all trips

feature_validation_df.select(
    "speed_difference",
    "fare_per_km_difference",
    "tip_ratio_difference"
).describe().show()

+-------+--------------------+----------------------+--------------------+
|summary|    speed_difference|fare_per_km_difference|tip_ratio_difference|
+-------+--------------------+----------------------+--------------------+
|  count|              120000|                120000|              120000|
|   mean|0.002274928841210...|  0.002497088601502...|1.447073914306076...|
| stddev|0.001569807042384...|  0.001453516756576476|1.652163350225022E-4|
|    min|                 0.0|                   0.0|                 0.0|
|    max|0.005000000000002558|  0.005000000000000782|5.000000000000004E-4|
+-------+--------------------+----------------------+--------------------+



In [0]:
# Analyse the distribution of feature discrepancies

feature_validation_df.agg(
    F.percentile_approx(
        "speed_difference",
        [0.50, 0.95, 0.99]
    ).alias("speed_percentiles"),

    F.percentile_approx(
        "fare_per_km_difference",
        [0.50, 0.95, 0.99]
    ).alias("fare_percentiles"),

    F.percentile_approx(
        "tip_ratio_difference",
        [0.50, 0.95, 0.99]
    ).alias("tip_percentiles")
).show(truncate=False)

+-------------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------------------------------+
|speed_percentiles                                                  |fare_percentiles                                                  |tip_percentiles                                                   |
+-------------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------------------------------+
|[0.002295081967213619, 0.004724409448822797, 0.0049794801641578346]|[0.0024999999999999467, 0.004758039338120579, 0.00495798319327756]|[6.796116504856342E-5, 4.566210045662128E-4, 4.910836762688497E-4]|
+-------------------------------------------------------------------+------------------------------------------------------------------+------------------------------------------------

#### Derived Feature Validation Findings

All three derived numerical features were independently
recalculated and validated across 120,000 records.

| Feature | Maximum Absolute Discrepancy |
|---|---:|
| Average speed | 0.005 km/h |
| Fare per kilometre | 0.005 EUR/km |
| Tip-to-fare ratio | 0.0005 |

The observed discrepancies are consistent with numerical
rounding and floating-point precision.

All original derived features are retained without
modification for subsequent analysis and anomaly detection.

### Categorical and Domain Validation

Assess categorical consistency, operational constraints
and geographical reference integrity to ensure that
the dataset is suitable for downstream analytical tasks.

In [0]:

# Inspect categorical distributions

categorical_columns = [
    "payment_method",
    "service_type",
    "dispatch_channel",
    "weather_condition",
    "pickup_day_of_week"
]

for column in categorical_columns:
    print(f"\nDistribution of {column}")

    trips_df.groupBy(column).count().orderBy(
        F.col("count").desc()
    ).show(truncate=False)


Distribution of payment_method
+-----------------+-----+
|payment_method   |count|
+-----------------+-----+
|Card             |74304|
|Cash             |23948|
|Mobile Wallet    |16913|
|Corporate Account|4835 |
+-----------------+-----+


Distribution of service_type
+------------+------+
|service_type|count |
+------------+------+
|Standard    |105608|
|Accessible  |8395  |
|Premium     |5997  |
+------------+------+


Distribution of dispatch_channel
+-----------------+-----+
|dispatch_channel |count|
+-----------------+-----+
|Mobile App       |75289|
|Street Hail      |21765|
|Corporate Booking|13190|
|Phone Dispatch   |9756 |
+-----------------+-----+


Distribution of weather_condition
+-----------------+-----+
|weather_condition|count|
+-----------------+-----+
|Clear            |51531|
|Cloudy           |33846|
|Rain             |22640|
|Heavy Rain       |8435 |
|Snow             |3548 |
+-----------------+-----+


Distribution of pickup_day_of_week
+------------------+-----

In [0]:
# Validate operational constraints

validation_rules = {
    "invalid_distance": F.col("trip_distance_km") <= 0,
    "invalid_duration": F.col("trip_duration_min") <= 0,
    "invalid_passenger_count": (
        (F.col("passenger_count") < 1) |
        (F.col("passenger_count") > 6)
    ),
    "invalid_pickup_hour": (
        (F.col("pickup_hour") < 0) |
        (F.col("pickup_hour") > 23)
    ),
    "invalid_average_speed": F.col("average_speed_kmh") <= 0,
    "invalid_fare_per_km": F.col("fare_per_km_eur") <= 0,
    "invalid_tip_ratio": F.col("tip_to_fare_ratio") < 0
}

trips_df.agg(*[
    F.sum(
        F.when(condition, 1).otherwise(0)
    ).alias(rule)
    for rule, condition in validation_rules.items()
]).show(truncate=False)

+----------------+----------------+-----------------------+-------------------+---------------------+-------------------+-----------------+
|invalid_distance|invalid_duration|invalid_passenger_count|invalid_pickup_hour|invalid_average_speed|invalid_fare_per_km|invalid_tip_ratio|
+----------------+----------------+-----------------------+-------------------+---------------------+-------------------+-----------------+
|0               |0               |0                      |0                  |0                    |0                  |0                |
+----------------+----------------+-----------------------+-------------------+---------------------+-------------------+-----------------+



In [0]:
# Identify string and floating-point columns

string_columns = [
    c for c, dtype in trips_df.dtypes
    if dtype == "string"
]

float_columns = [
    c for c, dtype in trips_df.dtypes
    if dtype in ["double", "float"]
]

# Check empty strings and whitespace

trips_df.agg(*[
    F.sum(
        F.when(F.trim(F.col(c)) == "", 1).otherwise(0)
    ).alias(c)
    for c in string_columns
]).show(truncate=False)

# Check numerical NaN values

trips_df.agg(*[
    F.sum(
        F.when(F.isnan(F.col(c)), 1).otherwise(0)
    ).alias(c)
    for c in float_columns
]).show(truncate=False)

+-------+---------+----------+-----------+------------+--------------+------------+----------------+-----------------+--------------+---------------+------------------+
|trip_id|driver_id|vehicle_id|pickup_zone|dropoff_zone|payment_method|service_type|dispatch_channel|weather_condition|scenario_label|scenario_detail|pickup_day_of_week|
+-------+---------+----------+-----------+------------+--------------+------------+----------------+-----------------+--------------+---------------+------------------+
|0      |0        |0         |0          |0           |0             |0           |0               |0                |0             |0              |0                 |
+-------+---------+----------+-----------+------------+--------------+------------+----------------+-----------------+--------------+---------------+------------------+

+----------------+-----------------+---------------+---------------+--------------+----------------+-----------------+---------------+-----------------+
|

In [0]:
# Load the geographical reference dataset

zones_path = (
    "/Volumes/workspace/default/urban_mobility/"
    "zone_reference.csv"
)

zones_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_path)
)

print("Total reference records:", zones_df.count())

zones_df.printSchema()

display(zones_df)

Total reference records: 20
root
 |-- zone_name: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- area_group: string (nullable = true)



zone_name,zone_type,area_group
Central Business District,Business,Core
Financial Quarter,Business,Core
University District,Education,Inner
Central Station,Transport,Core
Airport,Transport,Outer
North Residential,Residential,North
South Residential,Residential,South
West Shopping District,Retail,West
East Medical Campus,Healthcare,East
Convention Centre,Events,Core


In [0]:
# Validate uniqueness of geographical reference keys

zones_df.agg(
    F.count("*").alias("reference_records"),
    F.countDistinct("zone_name").alias("unique_zone_names"),
    F.sum(
        F.when(
            F.col("zone_name").isNull() |
            (F.trim(F.col("zone_name")) == ""),
            1
        ).otherwise(0)
    ).alias("invalid_zone_names")
).show()

+-----------------+-----------------+------------------+
|reference_records|unique_zone_names|invalid_zone_names|
+-----------------+-----------------+------------------+
|               20|               20|                 0|
+-----------------+-----------------+------------------+



In [0]:
# Identify trips without matching geographical references

invalid_pickup_zones = trips_df.join(
    zones_df,
    trips_df.pickup_zone == zones_df.zone_name,
    "left_anti"
)

invalid_dropoff_zones = trips_df.join(
    zones_df,
    trips_df.dropoff_zone == zones_df.zone_name,
    "left_anti"
)

print("Invalid pickup zones:", invalid_pickup_zones.count())
print("Invalid dropoff zones:", invalid_dropoff_zones.count())

Invalid pickup zones: 0
Invalid dropoff zones: 0


## Saving Validated Data.

In [0]:
# Persist validated trip records as a managed Delta table

trips_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.urban_mobility_trips_validated")

In [0]:
# Persist geographical reference data

zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.urban_mobility_zones")

In [0]:
spark.sql("""
    SHOW TABLES IN workspace.default
""").show(truncate=False)

+--------+------------------------------+-----------+
|database|tableName                     |isTemporary|
+--------+------------------------------+-----------+
|default |urban_mobility_trips_validated|false      |
|default |urban_mobility_zones          |false      |
+--------+------------------------------+-----------+

